In [1]:
import xarray as xr
import pandas as pd
from pathlib import Path
from dask import delayed, compute

# Use this notebook to create Zarr stores for each of the datasets
Should you wish not to use the default locations set by the tutorial notebook, you will have to provide the path to the netcdf files and the path to where you want the zarr store to exist.<br>

It is worth noting that each netcdf file contains all data for a single station, and there are two functions defined below that are necessary for converting our station data to zarr:
- `preprocess_station`
- `process_all_to_zarr`



There is a final function `load_combined_dataset` that is used to verify the conversion succeeded and that all station data can be loaded together.


# Preprocess NetCDF Files
Purpose:
- Opens a single NetCDF file, standardizes it, and prepares it for analysis.

Key steps:
- Loads the NetCDF file as an xarray Dataset.
- Drops the input_station_id variable if present (to avoid object dtype issues).
- Assigns a station_id coordinate from the file’s attributes or filename.
- Reindexes the time dimension to a common hourly range (ensuring all stations align in time).
- Returns the cleaned dataset.


In [2]:
def preprocess_station(file_path, date_range):
    """Open and preprocess a single NetCDF file."""
    ds = xr.open_dataset(file_path)
    
    # Clean invalid attributes
    #ds = clean_attrs(ds)

    if 'input_station_id' in ds:
        ds = ds.drop_vars('input_station_id')

    # Assign station ID from attributes or filename
    station_id = ds.attrs.get("station_id", file_path.stem)
    ds = ds.assign_coords(station_id=station_id)

    # Promote lat/lon/elevation to coordinates (if not already)
    for coord in ["latitude", "longitude", "elevation"]:
        if coord in ds and coord not in ds.coords:
            ds = ds.set_coords(coord)

    # Reindex time to common range
    target_time = pd.date_range(date_range[0], date_range[1], freq='h')
    ds = ds.reindex(time=target_time)

    return ds

    #Explain reason for reindexing and cutting out pre 1970 poor data quality
    

The HadISD NetCDF files store latitude, longitude, and elevation as coordinates with a singleton coordinate_length dimension. When merging multiple stations, these become coordinates of shape (station, coordinate_length). To ensure they are always available as coordinates (and not lost when selecting variables), we explicitly promote them with set_coords. After merging, we remove the unnecessary coordinate_length dimension, resulting in 1D auxiliary coordinates of shape (station,) for each station.

# Convert NetCDF to Zarr

Purpose:
- Batch-processes all NetCDF files in a directory, converting each to a Zarr store.

Key steps:
- Iterates through all .nc files in the input directory.
- Applies the preprocess_station function to each file.
- Saves each processed dataset as an individual Zarr store in the output directory.


In [3]:
def process_all_to_zarr(netcdf_dir, zarr_output_dir, date_range):
    """Convert NetCDF files to Zarr only if not already present in the Zarr directory."""
    netcdf_dir = Path(netcdf_dir)
    zarr_output_dir = Path(zarr_output_dir)
    zarr_output_dir.mkdir(parents=True, exist_ok=True)

    netcdf_files = list(netcdf_dir.glob("*.nc"))
    zarr_files = set(f.stem for f in zarr_output_dir.glob("*.zarr"))

    converted = 0
    skipped = 0

    for nc_file in netcdf_files:
        zarr_name = nc_file.stem
        out_path = zarr_output_dir / f"{zarr_name}.zarr"
        if zarr_name in zarr_files:
            print(f"Zarr file already exists for {nc_file.name}: {out_path.name}. Skipping.")
            skipped += 1
            continue
        print(f"Converting: {nc_file.name} → {out_path.name}")
        try:
            ds = preprocess_station(nc_file, date_range)
            ds.to_zarr(str(out_path), mode='w')
            converted += 1
        except Exception as e:
            print(f"Failed on {nc_file.name}: {e}")

    print(f"Conversion complete. {converted} new stations converted, {skipped} already present.")

In [4]:
def process_all_to_zarr(netcdf_dir, zarr_output_dir, date_range):
    """Convert NetCDF files to Zarr only if not already present in the Zarr directory."""
    netcdf_dir = Path(netcdf_dir)
    zarr_output_dir = Path(zarr_output_dir)
    zarr_output_dir.mkdir(parents=True, exist_ok=True)

    netcdf_files = list(netcdf_dir.glob("*.nc"))
    zarr_files = set(f.stem for f in zarr_output_dir.glob("*.zarr"))

    converted = 0
    skipped = 0

    for nc_file in netcdf_files:
        zarr_name = nc_file.stem
        out_path = zarr_output_dir / f"{zarr_name}.zarr"
        if zarr_name in zarr_files:
            print(f"Zarr file already exists for {nc_file.name}: {out_path.name}. Skipping.")
            skipped += 1
            continue
        print(f"Converting: {nc_file.name} → {out_path.name}")
        try:
            ds = preprocess_station(nc_file, date_range)
            ds.to_zarr(str(out_path), mode='w')
            converted += 1
        except Exception as e:
            print(f"Failed on {nc_file.name}: {e}")

    print(f"Conversion complete. {converted} new stations converted, {skipped} already present.")

# Load all individual Zarr stores into a single xarray Dataset
Purpose:
- Loads all individual Zarr stores and combines them into a single xarray Dataset for analysis.

Key steps:
- Finds all .zarr stores in the specified directory.
- Uses xr.open_mfdataset to open and concatenate them along the station dimension.
- Returns the combined dataset, ready for further analysis.

In [5]:
def load_combined_dataset(zarr_dir):
    # Open all Zarr stores together
    zarr_paths = list(Path(zarr_dir).glob("*.zarr"))

    # Combine along station dimension
    ds = xr.open_mfdataset(
        zarr_paths,
        combine="nested",
        concat_dim="station",
        parallel=True,
        engine="zarr"
    )
    return ds

Combining data like this is far quicker and more resource efficient than using NetCDF files directly.

# Execute the Pre-processing and Conversion to Zarr 
We can run the `Data_Config.ipynb` to set the following:
- Date range to reindex the data
- Path to NetCDFs that need converting to Zarr
- Output location of the Zarr store

In [6]:
%run Data_Config.ipynb
print(f"NetCDF input directory: {input_dir}")
print(f"Zarr output directory: {zarr_output_dir}")
print(f"Date range: {DATE_RANGE}")

NetCDF input directory: /Users/joelmiller/HadISD_data1/netcdf
Zarr output directory: /Users/joelmiller/HadISD_data1/zarr
Date range: ('1970-01-01T00', '2023-12-31T23')


In [7]:
process_all_to_zarr(str(input_dir), str(zarr_output_dir), DATE_RANGE)

Zarr file already exists for hadisd.3.4.0.2023f_19310101-20240101_085110-99999.nc: hadisd.3.4.0.2023f_19310101-20240101_085110-99999.zarr. Skipping.
Zarr file already exists for hadisd.3.4.0.2023f_19310101-20240101_720316-12984.nc: hadisd.3.4.0.2023f_19310101-20240101_720316-12984.zarr. Skipping.
Zarr file already exists for hadisd.3.4.0.2023f_19310101-20240101_080144-99999.nc: hadisd.3.4.0.2023f_19310101-20240101_080144-99999.zarr. Skipping.
Zarr file already exists for hadisd.3.4.0.2023f_19310101-20240101_080550-99999.nc: hadisd.3.4.0.2023f_19310101-20240101_080550-99999.zarr. Skipping.
Zarr file already exists for hadisd.3.4.0.2023f_19310101-20240101_085050-99999.nc: hadisd.3.4.0.2023f_19310101-20240101_085050-99999.zarr. Skipping.
Zarr file already exists for hadisd.3.4.0.2023f_19310101-20240101_720353-63875.nc: hadisd.3.4.0.2023f_19310101-20240101_720353-63875.zarr. Skipping.
Zarr file already exists for hadisd.3.4.0.2023f_19310101-20240101_082380-99999.nc: hadisd.3.4.0.2023f_1931

Show the combined dataset from the Zarr stores

In [8]:
ds_combined = load_combined_dataset(zarr_output_dir)
ds_combined

/Users/joelmiller/miniconda3/envs/pyearthtools/lib/python3.13/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/joelmiller/miniconda3/envs/pyearthtools/lib/python3.13/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/joelmiller/miniconda3/envs/pyearthtools/lib/python3.13/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/joelmiller/miniconda3/envs/pyearthto

<xarray.Dataset> Size: 89GB
Dimensions:                (station: 210, time: 473352, flagged: 19,
                            reporting_v: 19, reporting_t: 1116, reporting_2: 2,
                            test: 71, coordinate_length: 1)
Coordinates:
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-3...
    longitude              (station, coordinate_length) float64 2kB dask.array<chunksize=(1, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 2kB dask.array<chunksize=(1, 1), meta=np.ndarray>
    elevation              (station, coordinate_length) float64 2kB dask.array<chunksize=(1, 1), meta=np.ndarray>
    station_id             (station) object 2kB '720328-63832' ... '720361-63...
Dimensions without coordinates: station, flagged, reporting_v, reporting_t,
                                reporting_2, test, coordinate_length
Data variables: (12/25)
    flagged_obs            (station, time, flagged) float64 15GB dask.array<chunksize=(1, 29585, 3), meta=np.ndarray>
    total_cloud_cover      (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    slp                    (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    precip12_depth         (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    high_cloud_cover       (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    precip1_depth          (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    ...                     ...
    precip18_depth         (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    precip9_depth          (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    quality_control_flags  (station, time, test) float64 56GB dask.array<chunksize=(1, 29585, 5), meta=np.ndarray>
    wind_gust              (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    winddirs               (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    precip24_depth         (station, time) float64 795MB dask.array<chunksize=(1, 59169), meta=np.ndarray>
Attributes: (12/39)
    title:                       HadISD
    institution:                 Met Office Hadley Centre, Exeter, UK
    source:                      HadISD data product
    references:                  Dunn, 2019, Met Office Hadley Centre Technic...
    creator_name:                Robert Dunn
    creator_url:                 www.metoffice.gov.uk
    ...                          ...
    station_information:         Where station is a composite the station id ...
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    featureType:                 timeSeries
    processing_date:             08-Jan-2024
    history:                     Created by mk_netcdf_files.py \nDuplicate Mo...

# Data Organization: NetCDF and Zarr stores

To keep your workflow clear and reproducible, we recommend storing both the raw NetCDF files and the processed Zarr data in separate subfolders inside your main WMO directory. For example:

- `HadISD_data/WMO_080000-099999/netcdf/` (raw NetCDF files)
- `HadISD_data/WMO_080000-099999/zarr/` (processed Zarr stores with harmonized time coordinates)

This makes it obvious which data is raw and which is ready for fast, parallel analysis.